# Координаты и межгодовое сопоставление: диагностический аудит

Graph29, 2026-10-05. Аудит не принимает точки или связи. Единица проверки — выбранная запись источника. Одинаковая точка является сигналом проверки, а не доказательством неверности всех записей группы.

Полный выполненный расчёт: `../../mass_linkage/audit_coordinate_proximity_premise_20261005.py`; результаты и ограничения: `README.md`. Ноутбук содержит проверку входов и ключевой SQL. Ожидаемые результаты взяты из сохранённого receipt; этот ноутбук отдельно не исполнялся.

In [ ]:
from pathlib import Path
import hashlib, json
import duckdb

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'research_rebuild/mass_linkage/build_long_table.py').exists())
BASE = REPO / 'research_rebuild/evidence/coordinate_proximity_audit_20261005'
receipt = json.loads((BASE / 'receipt.json').read_text())
for key, item in receipt['input_files'].items():
    with Path(item['path']).open('rb') as f:
        assert hashlib.file_digest(f, 'sha256').hexdigest() == item['sha256'], key
print(receipt['status'], receipt['cardinality'])


## Буквально одинаковые точки внутри одного года

Используются точные разрешённые admission statuses сохранённого расчёта. Ретроспективные representative points включены; их историческая точность этим запросом не доказывается.

In [ ]:
con = duckdb.connect(config={'threads': 2, 'memory_limit': '1GB'})
for key in ('selected', 'points'):
    con.read_parquet(receipt['input_files'][key]['path']).create_view(key)
allowlist = ','.join("'" + x + "'" for x in receipt['point_status_allowlist'])
query = f"""
WITH accepted AS (
    SELECT s.census_year, s.source_record_id, s.population, p.latitude, p.longitude
    FROM selected s JOIN points p ON p.target_source_record_id=s.source_record_id
    WHERE p.coordinate_admission_status IN ({allowlist})
), groups AS (
    SELECT census_year, latitude, longitude, count(*) n, sum(population) pop
    FROM accepted GROUP BY 1,2,3 HAVING count(*)>1
)
SELECT census_year, count(*) collision_groups, sum(n) affected_rows,
       sum(pop) selected_population FROM groups GROUP BY 1 ORDER BY 1
"""
actual = con.execute(query).fetchall()
expected = [(r['census_year'], r['collision_groups'], r['affected_rows'], r['selected_population']) for r in receipt['accepted_same_year_collisions']]
assert actual == expected
actual


## Межгодовые кандидаты

Точное нормализованное имя + тип + регион; радиус 1 км — диагностический порог. Полный граф используется для проверки компонент и повторных лет. Нет проверки всех событий/holds, нет новых принятых решений. Независимый расчёт spherical KDTree находится в `independent_candidate_readback.py` и согласуется с основным SQL для всех трёх пар лет.

In [ ]:
import pandas as pd
experiment = pd.DataFrame(receipt['exact_name_proximity_experiment'])
display(experiment[experiment.radius_m.eq(1000)])
print(receipt['new_pair_batch_preflight'])
pairs = pd.read_csv(BASE / 'new_component_compatible_pairs_within1km.csv')
assert len(pairs) == 773
assert not pairs.from_same_year_point_collision.any()
assert not pairs.to_same_year_point_collision.any()
pairs.head()


## Полный повторный запуск

Запускать из корня checkout, выходной каталог должен быть новым:

```sh
python research_rebuild/mass_linkage/audit_coordinate_proximity_premise_20261005.py --out /tmp/coordinate-proximity-audit-new-run
```

Скрипт проверяет хеши всех входов до/после, сохраняет CSV и receipt. Не использовать совпадение перенесённых по связи координат как независимое подтверждение той же связи.